# E3: LSTM, GRU trên chuỗi ảnh Fashion-MNIST

Notebook này chạy toàn bộ pipeline của bài E3 trên Colab. Cần bật GPU bằng cách vào Runtime, chọn Change runtime type, rồi chọn GPU.

E3 dùng lại đúng dataset, split và checkpoint MLP, CNN đã có từ E1, nên nhánh được clone phải đã chứa cả thư mục E1 với kết quả đã train, không chỉ thư mục E3. Nếu đang làm việc trên một nhánh khác main thì sửa biến BRANCH ở cell bên dưới cho đúng tên nhánh đó.

In [ ]:
!nvidia-smi

## 1. Clone repo và chuyển vào thư mục E3

In [ ]:
import os

REPO_URL = "https://github.com/truong1501/SDH_HK261_CO5085.git"
REPO_DIR = "/content/SDH_HK261_CO5085"
BRANCH = "main"  # doi lai dung nhanh dang chua thu muc E3/

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git checkout {BRANCH} && git pull

%cd {REPO_DIR}/E3

## 2. Cài đặt thư viện

Máy ảo của Colab đã có sẵn torch và torchvision tương thích với GPU đang chạy, không cần cài thêm gì cho E3.

In [ ]:
import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())

## 3. Kiểm tra nhanh sequences và models

In [ ]:
!python -m src.sequences
!python -m src.models

## 4. Huấn luyện bốn run: lstm/gru kết hợp với row/patch

In [ ]:
!python -m src.train --model lstm --seq row

In [ ]:
!python -m src.train --model gru --seq row

In [ ]:
!python -m src.train --model lstm --seq patch

In [ ]:
!python -m src.train --model gru --seq patch

## 5. Đánh giá lại baseline MLP, CNN từ E1

Bước này không huấn luyện lại, chỉ load checkpoint đã có sẵn trong
`../E1/results/{mlp,cnn}/best.pt` và chạy trên cùng tập test. Nếu thư mục đó
chưa có checkpoint, cần chạy notebook của E1 trước.

In [ ]:
!python -m src.baseline

## 6. So sánh toàn bộ sáu dòng: bốn run recurrent và hai baseline

In [ ]:
!python -m src.evaluate

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/comparison_table.csv"))
display(Image("results/loss_accuracy_curves.png"))

## 7. Lưu kết quả

Thư mục results chứa checkpoint của từng run, log huấn luyện theo từng epoch,
bảng so sánh và biểu đồ, dùng để viết phần báo cáo của E3. Có thể chép thư
mục này vào Google Drive hoặc nén lại rồi tải thẳng về máy bằng hai cell bên
dưới.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/CO5085_E3_results
!cp -r results /content/drive/MyDrive/CO5085_E3_results/

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/E3_results", "zip", "results")
files.download("/content/E3_results.zip")